In [1]:
import os

os.environ["HADOOP_HOME"] = r"C:\Users\91840\OneDrive\Documents\hadoop-win-utils"
os.environ["PATH"] += os.pathsep + os.path.join(
    os.environ["HADOOP_HOME"],
    "bin"
)

Import libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

Start Spark

In [3]:
spark = (
    SparkSession.builder
    .appName("SupportIQ-Transformation")
    .master("local[*]")
    .getOrCreate()
)

c:\Users\91840\OneDrive\Documents\PYTHON PROJECTS\support-iq\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Define the cleaned-data path

In [4]:
clean_path = "../data/processed/support_tickets_cleaned"

Load all Parquet files

In [5]:
parquet_files = [

    os.path.join(clean_path, file)

    for file in os.listdir(clean_path)

    if file.endswith(".parquet")

]

In [6]:
print(
    "Number of Parquet files found:",
    len(parquet_files)
)

tickets_df = spark.read.parquet(
    *parquet_files
)

print(
    "All Parquet files loaded successfully."
)

Number of Parquet files found: 9
All Parquet files loaded successfully.


Create created_year

In [7]:
tickets_df = tickets_df.withColumn(
    "created_year",
    F.year(
        F.col("created_date")
    )
)

Check


In [8]:
tickets_df.select(
    "ticket_id",
    "created_date",
    "created_year"
).show(
    10,
    truncate=False
)

+----------+------------+------------+
|ticket_id |created_date|created_year|
+----------+------------+------------+
|TKT0000141|2025-02-06  |2025        |
|TKT0000404|2025-01-10  |2025        |
|TKT0000479|2026-01-21  |2026        |
|TKT0000603|2025-08-30  |2025        |
|TKT0000984|2025-11-18  |2025        |
|TKT0001265|2025-04-24  |2025        |
|TKT0001548|2025-07-19  |2025        |
|TKT0001974|2025-09-14  |2025        |
|TKT0001982|2025-06-06  |2025        |
|TKT0001992|2025-08-22  |2025        |
+----------+------------+------------+
only showing top 10 rows


Create created_month

In [9]:
tickets_df = tickets_df.withColumn(
    "created_month",
    F.month(
        F.col("created_date")
    )
)

Check

In [10]:
tickets_df.select(
    "ticket_id",
    "created_date",
    "created_year",
    "created_month"
).show(
    10,
    truncate=False
)

+----------+------------+------------+-------------+
|ticket_id |created_date|created_year|created_month|
+----------+------------+------------+-------------+
|TKT0000141|2025-02-06  |2025        |2            |
|TKT0000404|2025-01-10  |2025        |1            |
|TKT0000479|2026-01-21  |2026        |1            |
|TKT0000603|2025-08-30  |2025        |8            |
|TKT0000984|2025-11-18  |2025        |11           |
|TKT0001265|2025-04-24  |2025        |4            |
|TKT0001548|2025-07-19  |2025        |7            |
|TKT0001974|2025-09-14  |2025        |9            |
|TKT0001982|2025-06-06  |2025        |6            |
|TKT0001992|2025-08-22  |2025        |8            |
+----------+------------+------------+-------------+
only showing top 10 rows


Create resolution_category

In [11]:
tickets_df = tickets_df.withColumn(
    "resolution_category",
    F.when(
        F.col("resolution_hours").isNull(),
        "Unknown"
    )
    .when(
        F.col("resolution_hours") <= 24,
        "Within 1 Day"
    )
    .when(
        F.col("resolution_hours") <= 48,
        "1-2 Days"
    )
    .when(
        F.col("resolution_hours") <= 72,
        "2-3 Days"
    )
    .otherwise(
        "More Than 3 Days"
    )
)

Check

In [12]:
tickets_df.select(
    "ticket_id",
    "resolution_hours",
    "resolution_category"
).show(
    20,
    truncate=False
)

+----------+----------------+-------------------+
|ticket_id |resolution_hours|resolution_category|
+----------+----------------+-------------------+
|TKT0000141|74.8            |More Than 3 Days   |
|TKT0000404|155.93          |More Than 3 Days   |
|TKT0000479|34.04           |1-2 Days           |
|TKT0000603|91.77           |More Than 3 Days   |
|TKT0000984|151.56          |More Than 3 Days   |
|TKT0001265|164.82          |More Than 3 Days   |
|TKT0001548|107.73          |More Than 3 Days   |
|TKT0001974|128.37          |More Than 3 Days   |
|TKT0001982|92.02           |More Than 3 Days   |
|TKT0001992|75.98           |More Than 3 Days   |
|TKT0002177|25.15           |1-2 Days           |
|TKT0002182|194.53          |More Than 3 Days   |
|TKT0002424|150.95          |More Than 3 Days   |
|TKT0002525|166.45          |More Than 3 Days   |
|TKT0002610|42.18           |1-2 Days           |
|TKT0002915|144.19          |More Than 3 Days   |
|TKT0002970|103.68          |More Than 3 Days   |


Create first_response_category

In [13]:
tickets_df = tickets_df.withColumn(
    "first_response_category",
    F.when(
        F.col("first_response_minutes").isNull(),
        "Unknown"
    )
    .when(
        F.col("first_response_minutes") <= 30,
        "Very Fast"
    )
    .when(
        F.col("first_response_minutes") <= 60,
        "Fast"
    )
    .when(
        F.col("first_response_minutes") <= 120,
        "Moderate"
    )
    .otherwise(
        "Slow"
    )
)

Check

In [14]:
tickets_df.select(
    "ticket_id",
    "first_response_minutes",
    "first_response_category"
).show(
    20,
    truncate=False
)

+----------+----------------------+-----------------------+
|ticket_id |first_response_minutes|first_response_category|
+----------+----------------------+-----------------------+
|TKT0000141|500                   |Slow                   |
|TKT0000404|76                    |Moderate               |
|TKT0000479|530                   |Slow                   |
|TKT0000603|67                    |Moderate               |
|TKT0000984|517                   |Slow                   |
|TKT0001265|209                   |Slow                   |
|TKT0001548|372                   |Slow                   |
|TKT0001974|186                   |Slow                   |
|TKT0001982|466                   |Slow                   |
|TKT0001992|554                   |Slow                   |
|TKT0002177|134                   |Slow                   |
|TKT0002182|230                   |Slow                   |
|TKT0002424|319                   |Slow                   |
|TKT0002525|52                    |Fast 

Create satisfaction_category

In [15]:
tickets_df = tickets_df.withColumn(
    "satisfaction_category",
    F.when(
        F.col("customer_satisfaction").isNull(),
        "Not Rated"
    )
    .when(
        F.col("customer_satisfaction") <= 2,
        "Dissatisfied"
    )
    .when(
        F.col("customer_satisfaction") == 3,
        "Neutral"
    )
    .otherwise(
        "Satisfied"
    )
)

Check

In [16]:
tickets_df.select(
    "ticket_id",
    "customer_satisfaction",
    "satisfaction_category"
).show(
    20,
    truncate=False
)

+----------+---------------------+---------------------+
|ticket_id |customer_satisfaction|satisfaction_category|
+----------+---------------------+---------------------+
|TKT0000141|1.0                  |Dissatisfied         |
|TKT0000404|1.0                  |Dissatisfied         |
|TKT0000479|3.0                  |Neutral              |
|TKT0000603|2.0                  |Dissatisfied         |
|TKT0000984|1.0                  |Dissatisfied         |
|TKT0001265|3.0                  |Neutral              |
|TKT0001548|2.0                  |Dissatisfied         |
|TKT0001974|4.0                  |Satisfied            |
|TKT0001982|1.0                  |Dissatisfied         |
|TKT0001992|NULL                 |Not Rated            |
|TKT0002177|3.0                  |Neutral              |
|TKT0002182|1.0                  |Dissatisfied         |
|TKT0002424|5.0                  |Satisfied            |
|TKT0002525|4.0                  |Satisfied            |
|TKT0002610|1.0                

Create escalation_indicator

In [17]:
tickets_df = tickets_df.withColumn(
    "escalation_indicator",
    F.when(
        F.col("escalation_flag") == "Yes",
        1
    )
    .otherwise(
        0
    )
)

Check

In [18]:
tickets_df.select(
    "ticket_id",
    "escalation_flag",
    "escalation_indicator"
).show(
    20,
    truncate=False
)

+----------+---------------+--------------------+
|ticket_id |escalation_flag|escalation_indicator|
+----------+---------------+--------------------+
|TKT0000141|Yes            |1                   |
|TKT0000404|No             |0                   |
|TKT0000479|Yes            |1                   |
|TKT0000603|No             |0                   |
|TKT0000984|No             |0                   |
|TKT0001265|No             |0                   |
|TKT0001548|Yes            |1                   |
|TKT0001974|Yes            |1                   |
|TKT0001982|No             |0                   |
|TKT0001992|Yes            |1                   |
|TKT0002177|No             |0                   |
|TKT0002182|No             |0                   |
|TKT0002424|Yes            |1                   |
|TKT0002525|No             |0                   |
|TKT0002610|No             |0                   |
|TKT0002915|No             |0                   |
|TKT0002970|Yes            |1                   |


Create a priority_level

In [19]:
tickets_df = tickets_df.withColumn(
    "priority_level",
    F.when(
        F.col("priority") == "Low",
        1
    )
    .when(
        F.col("priority") == "Medium",
        2
    )
    .when(
        F.col("priority") == "High",
        3
    )
    .when(
        F.col("priority") == "Critical",
        4
    )
    .otherwise(
        0
    )
)

Check

In [20]:
tickets_df.select(
    "ticket_id",
    "priority",
    "priority_level"
).show(
    20,
    truncate=False
)

+----------+--------+--------------+
|ticket_id |priority|priority_level|
+----------+--------+--------------+
|TKT0000141|Medium  |2             |
|TKT0000404|Critical|4             |
|TKT0000479|High    |3             |
|TKT0000603|High    |3             |
|TKT0000984|High    |3             |
|TKT0001265|Critical|4             |
|TKT0001548|High    |3             |
|TKT0001974|Medium  |2             |
|TKT0001982|Low     |1             |
|TKT0001992|High    |3             |
|TKT0002177|High    |3             |
|TKT0002182|Critical|4             |
|TKT0002424|High    |3             |
|TKT0002525|Unknown |0             |
|TKT0002610|Low     |1             |
|TKT0002915|High    |3             |
|TKT0002970|Low     |1             |
|TKT0002987|Unknown |0             |
|TKT0003376|Low     |1             |
|TKT0003599|High    |3             |
+----------+--------+--------------+
only showing top 20 rows


Save the transformed data

In [21]:
transformed_path = "../data/processed/support_tickets_transformed"

os.makedirs(
    "../data/processed",
    exist_ok=True
)

In [22]:
tickets_df.write \
    .mode("overwrite") \
    .parquet(transformed_path)

Load the saved transformed data

In [23]:
transformed_parquet_files = [

    os.path.join(
        transformed_path,
        file
    )

    for file in os.listdir(
        transformed_path
    )

    if file.endswith(".parquet")

]

In [24]:
print(
    "Number of transformed Parquet files:",
    len(transformed_parquet_files)
)

Number of transformed Parquet files: 8


In [25]:
transformed_df = spark.read.parquet(
    *transformed_parquet_files
)

In [26]:
print(
    "Transformed data loaded successfully."
)

print(
    "Rows:",
    transformed_df.count()
)

print(
    "Columns:",
    len(transformed_df.columns)
)

Transformed data loaded successfully.
Rows: 1204092
Columns: 25
